# Notebook 16: Deploying Agents with LangServe

**🎯 Goal:** Learn how to take a LangChain agent or chain and deploy it as a production-ready REST API using LangServe.

## 🧩 Concept Overview

Once you've built your amazing AI agent, you need a way for other applications (like a web frontend or a mobile app) to use it. This is typically done by exposing your agent as a REST API.

**LangServe** is a part of the LangChain ecosystem that makes this incredibly easy. It takes any LangChain 'runnable' (like a chain or a compiled LangGraph app) and automatically creates a robust, production-ready API for it.

Key features of LangServe include:
- **Automatic API Endpoints:** Creates endpoints for `invoke`, `batch`, and `stream` out of the box.
- **Interactive Playground:** Provides a web UI for playing with your deployed agent directly in the browser.
- **Built-in Streaming:** Natively supports streaming intermediate steps, which is great for user-facing applications.

## 🖼️ Visual Diagram: LangServe Architecture

```
 +----------+      POST /assistant/invoke      +----------------+
 | Web App  | ------------------------------> | FastAPI Server |
 | (Client) |                                 |                | ----> (Runs Agent)
 +----------+      <-------------------------- |  (LangServe)   |
                   JSON Response               +----------------+
```

## ⚙️ Setup

To deploy our agent, we need `langserve` and a web server library like `fastapi` and `uvicorn`.

In [ ]:
# pip install langserve fastapi uvicorn

# For this notebook, we'll first re-create the simple agent from Notebook 3
# to have a clean object to deploy.
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def get_time(query: str = "") -> str:
    from datetime import datetime
    return datetime.now().strftime("%H:%M")

tools = [Tool(name="Clock", func=get_time, description="Returns the current time.")]
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agent = create_tool_calling_agent(llm, tools, prompt)
agent_to_deploy = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("Agent to be deployed is ready.")

## 1. Creating the Server File

The standard way to deploy with LangServe is to create a `server.py` file. This file will:
1.  Import your agent application.
2.  Import `FastAPI` and `add_routes`.
3.  Use `add_routes` to attach your agent to a specific API path.

We will use a magic command to write the following code into a file named `server.py`.

In [ ]:
%%writefile server.py
from fastapi import FastAPI
from langserve import add_routes
from typing import Dict, Any

# --- 1. Import your agent from another file ---
# For this example, we'll redefine a simple agent here.
# In a real project, you would import `agent_to_deploy` from your notebook/module.
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o", temperature=0)
def get_time(query: str = "") -> str:
    from datetime import datetime
    return datetime.now().strftime("%H:%M")
tools = [Tool(name="Clock", func=get_time, description="Returns the current time.")]
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agent = create_tool_calling_agent(llm, tools, prompt)
agent_to_deploy = AgentExecutor(agent=agent, tools=tools, verbose=True)

# --- 2. Create the FastAPI app ---
app = FastAPI(
    title="My LangChain Agent Server",
    version="1.0",
    description="A simple API server for a LangChain agent.",
)

# --- 3. Add the LangServe routes ---
add_routes(
    app,
    agent_to_deploy,
    path="/agent", # The path where the agent will be hosted
)

# --- 4. Add a main block to run the server ---
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


## 2. Running the Server (Conceptual)

Because we cannot run a persistent server in this notebook environment, this step is conceptual. In your own terminal, you would run the following command:

```bash
python server.py
```

You would see output from `uvicorn` indicating the server is running:
```
INFO:     Started server process [12345]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
```

## 3. Interacting with the Deployed Agent

Once the server is running, you can interact with it using any HTTP client, like `curl` or Python's `requests` library.

### Using `curl`

```bash
curl -X POST http://localhost:8000/agent/invoke \
-H 'Content-Type: application/json' \
-d '{
  "input": {"input": "What time is it?"}
}'
```

### Using Python `requests`

In [ ]:
import requests
import json

response = requests.post(
    "http://localhost:8000/agent/invoke", # Assuming the server is running locally
    json={"input": {"input": "What time is it?"}}
)

# try:
#     result = response.json()
#     print(result['output'])
# except requests.exceptions.ConnectionError:
#     print("Could not connect to the server. Please run 'python server.py' in a separate terminal.")
print("This cell shows how to programmatically call your deployed agent.")

## 4. The LangServe Playground

One of the best features of LangServe is the built-in web interface for playing with your agent. Once your server is running, navigate to:

**http://localhost:8000/agent/playground/**

You'll find a user-friendly UI where you can enter inputs and see the agent's responses and intermediate steps, which is incredibly useful for testing and demonstration.

## 💡 Pro Tip

LangServe automatically creates several useful endpoints for your application. The most important one besides `/invoke` is `/stream`. By sending a request to `/agent/stream`, you can get a real-time stream of the agent's internal steps, which is perfect for building responsive frontends that show the agent's thought process as it happens.

## 🏁 Series Finale: Summary

Congratulations! You've reached the end of the expanded LangChain Learning Series. You now have the complete toolkit to not only build sophisticated AI agents but also to deploy them as robust, production-ready services.

1.  **LangServe Deploys Runnables:** It can take any LangChain object that is a 'runnable' (chains, agents, graphs) and expose it as an API.
2.  **Deployment is Simple:** Creating a `server.py` file with FastAPI and `add_routes` is all it takes to get a production-ready server running.
3.  **You Get More Than Just an API:** LangServe provides a rich feature set out of the box, including a web playground, streaming, and batching, which accelerates development and improves the end-user experience.

From here, the possibilities are endless. You can connect your deployed agents to custom frontends, integrate them into larger applications, and build truly powerful AI-powered products. Happy building!